# Abstractive Dialogue Summarization with T5 Transformer
### End-to-End Fine-Tuning, Evaluation, and Production Serving on the SAMSum Dataset
**Author:** Bhavesh Srivastava | **Architecture:** Google T5 (Text-to-Text Transfer Transformer)

## 1. Problem Formulation & Theoretical Background

### Why Dialogue Summarization?
Unlike standard document summarization (e.g., CNN/DailyMail news articles) which features structured paragraphs and high extractive bias, **dialogue summarization** poses unique natural language challenges:
- **Multi-turn conversational flow:** Information is fragmented across alternating speaker utterances.
- **Colloquial and informal speech:** Slang, abbreviations, incomplete sentences, and typos.
- **Coreference resolution:** Pronouns (*he, she, they, that*) require cross-turn context tracking.
- **Abstractive necessity:** Dialogue cannot be extracted verbatim; it must be synthesized into a coherent third-person narrative.

### T5 Architecture (Text-to-Text Transfer Transformer)
T5 (Raffel et al., 2020) reframes all NLP tasks into a unified text-to-text format. It employs a standard encoder-decoder Transformer with:
- **Encoder:** Encodes input conversational context with bidirectional self-attention and relative positional embeddings.
- **Decoder:** Autoregressively generates target summary tokens using causal self-attention and encoder-decoder cross-attention.
- **Relative Position Bias:** Replaces fixed sinusoidal position embeddings with learned bucketed relative positional biases added directly to attention logits: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + B_{\text{rel}}\right)V$.

## 2. Environment Setup & Library Imports

In [59]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments,  T5ForConditionalGeneration

In [60]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [61]:
import os
# Robust path resolution across local and Colab environments
data_dir = 'data' if os.path.exists('data') else ('../data' if os.path.exists('../data') else '.')
train_path = os.path.join(data_dir, 'samsum-train.csv')
val_path = os.path.join(data_dir, 'samsum-validation.csv')

if not os.path.exists(train_path):
    train_path = '/content/drive/MyDrive/Datsets/Text_Summarizer/samsum-train.csv'
    val_path = '/content/drive/MyDrive/Datsets/Text_Summarizer/samsum-validation.csv'

train_data = pd.read_csv(train_path)
val_data = pd.read_csv(val_path)
print(f'Loaded train: {train_data.shape}, val: {val_data.shape}')

In [62]:
# !ls /content/drive/MyDrive/Datsets/Text_Summarizer/

In [63]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [64]:
import torch
print(torch.cuda.get_device_name(0)) if torch.cuda.is_available() else print("No GPU")


Tesla T4


In [65]:
## -DownSizing the data for easier perocessing
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [66]:
print(train_data.shape)
print(val_data.shape)

(4000, 3)
(500, 3)


## 3. Data Ingestion & Preprocessing

We utilize the **SAMSum corpus** (~16k chat-style dialogues created by linguistic annotators). We downsample to 4,000 training and 500 validation dialogues for rapid iteration while preserving high linguistic diversity.

### Preprocessing Strategy
1. Normalize CRLF and consecutive whitespace.
2. Strip residual HTML tags.
3. Case-normalize to lowercase to match standard T5 lowercased subword distributions.

In [67]:
import re

def clean_text(text):
    text=re.sub(r"\r\n"," ",text) ##next Lines
    text=re.sub(r"\s+"," ",text)  ## Empty spaces
    text=re.sub(r"<.*?>"," ",text) ## HTML tags
    return text.strip().lower()


In [68]:
## Cleaning the train and validation data
train_data["dialogue"]=train_data["dialogue"].apply(clean_text)
train_data["summary"]=train_data["summary"].apply(clean_text) 

val_data["dialogue"]=val_data["dialogue"].apply(clean_text)
val_data["summary"]=val_data["summary"].apply(clean_text) 

In [69]:
train_data["dialogue"][2]

"jane:   jane: whaddya think? shona: this ur tinder profile thing? jane: yeah, i'm updating my profile tonite. kinda nervoous though... :( jane: what if i get another guy like john? o.o shona: john was a dickhead jane: preach sistah! shona: anyhoo - this time i've got u :d no slimeballs for you jane: not again *shudders* jane: you know he forgot my birthday??!! shona: wanker"

## 4. SentencePiece Subword Tokenization

T5 utilizes a **SentencePiece unigram tokenizer** with a vocabulary size of 32,128 tokens.
- **Input Context Length:** Truncated and padded to 512 tokens.
- **Target Summary Length:** Truncated and padded to 140 tokens.
- **Labels Alignment:** Decoder targets are aligned as `labels` in the training dictionary.

In [70]:
import os
from huggingface_hub import login

# Authenticate with Hugging Face Hub (optional for gated/private repositories)
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Authenticated with Hugging Face Hub successfully.")
else:
    print("HF_TOKEN environment variable not set. Continuing with local/public assets.")

In [71]:
## We will tokenize the raw inpput so that it can be fed to the model
## We will use the T-5's inbuilt vocabulary 
tokenizer=T5Tokenizer.from_pretrained("t5-small")

def tokenize(data):
    inputs=tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets=tokenizer(data["summary"],padding="max_length",max_length=140,truncation=True)
    inputs["labels"]=targets["input_ids"]  ## merging the feature tokens and target tokens

    return inputs

In [72]:
train_dataset=train_data.apply(tokenize,axis=1).to_list()
val_dataset=val_data.apply(tokenize,axis=1).to_list()

In [106]:
print(train_dataset[0])
## Input ids (dialogue)=> token id 
## Token Value 1 detrmines end of sequence and 0 determine the padding 
## Attention masks give us the valid token ids(1) and padding(0)  indexing
## Labes- target =>summary

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [74]:
len(train_dataset[0]["input_ids"])

512

In [75]:
print(type(train_dataset))

<class 'list'>


## 5. Model Initialization & Seq2Seq Fine-Tuning Setup

We load pre-trained `t5-small` (~60M parameters, 6 encoder layers, 6 decoder layers, 8 attention heads, $d_{\text{model}}=512$, $d_{\text{ff}}=2048$).

### Optimization Configuration
- **Loss Function:** Sequence-to-Sequence Cross-Entropy Loss with teacher forcing.
- **Optimizer:** AdamW with weight decay $\lambda = 0.01$.
- **Warmup:** 500 steps linear learning rate warmup.
- **Batch Size:** 8 per device, evaluated at each epoch boundary.

In [76]:
model=T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [77]:
## Selecting the hardware over which we need to train (CPU,GPU,TPU)

if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")

print("Device: ",device)
model.to(device)

Device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
## Fine Tuning the model

training_arguments=TrainingArguments(
    output_dir="./results",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500

)

In [79]:
trainer=Trainer(
    model=model,
    args=training_arguments,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [80]:
## Training the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.689652,0.409222
2,0.424491,0.384984
3,0.399850,0.380439
4,0.387765,0.375530
5,0.380128,0.374005
6,0.375550,0.373915


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9429061940511068, metrics={'train_runtime': 1125.5731, 'train_samples_per_second': 21.322, 'train_steps_per_second': 2.665, 'total_flos': 3248203235328000.0, 'train_loss': 0.9429061940511068, 'epoch': 6.0})

In [90]:
## moving the results models from colab instance to the local machine
!cp -r /content/results /content/drive/MyDrive



## 6. Model Serialization & Export

We save both the fine-tuned model checkpoint and tokenizer configuration for zero-dependency inference serving.

In [91]:
model.save_pretrained("/content/drive/MyDrive/Datsets/Text_Summarizer/saved_summary_model")
tokenizer.save_pretrained("/content/drive/MyDrive/Datsets/Text_Summarizer/saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Datsets/Text_Summarizer/saved_summary_model/tokenizer_config.json',
 '/content/drive/MyDrive/Datsets/Text_Summarizer/saved_summary_model/tokenizer.json')

In [93]:
## Loading the Saved Model And the Tokenizer for Evaluating it
model=T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/Datsets/Text_Summarizer/saved_summary_model")
tokenizer=T5Tokenizer.from_pretrained("/content/drive/MyDrive/Datsets/Text_Summarizer/saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## 7. Inference Pipeline & Beam Search Generation

### Decoding Strategy: Beam Search (`num_beams=4`)
Rather than greedy decoding (which selects the single highest-probability token at each step and risks local optima), **Beam Search** maintains the top-$k$ hypotheses with the highest cumulative log-probabilities:
$$\mathcal{S}^* = \arg\max_{\mathcal{Y}} \sum_{t=1}^T \log P(y_t \mid y_{<t}, X)$$
Combined with `early_stopping=True`, this guarantees concise, grammatically coherent summaries.

In [101]:
def summarize_dialogue(dialogue):
    # Cleaning
    dialogue=clean_text(dialogue)

    # Tokenizing
    input=tokenizer(dialogue,padding="max_length",max_length=512,truncation=True,return_tensors="pt").to(device)

    # Prediction(summary) in the token id form
    model.to(device)
    targets=model.generate(
        input_ids=input["input_ids"],
        attention_mask=input["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # Decoding the Target Tokens
    summary=tokenizer.decode(targets[0],skip_special_tokens=True)
    return summary

In [103]:
### Testing on a sample Dialogue

test_dialogue="""
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary=summarize_dialogue(test_dialogue)
print("Summary: ",summary)

Summary:  ai adoption has significantly increased over the past few years. ai systems are becoming more capable due to advances in deep learning and access to large datasets. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.
